# Evaluate CaRL × Luna experiments

This notebook reads **saved results only**. It builds editable `plotly.graph_objects` figures, tables, calibration diagnostics and board views; it never performs model inference. The default inputs are **mock interface checks**, not evidence of Luna accuracy or speed.

Use the project `.venv/bin/python` kernel and run top to bottom. If the mock inputs do not yet exist, run `make icl-eval-demo` from CaRL (offline). For a real run, set `RUN_DIRS` to one or more saved matrix/condition directories and `OUTPUT_DIR` to a new evaluation folder. Required files per condition are `config.json`, `results.json` and the individual detail JSONs.

## 1. Setup and select saved inputs

Conditions are grouped into cohorts by provider, policy fingerprint, model, board family, split, difficulty and experiment settings. Relative and fixed methods may be compared when the other settings match. Overlapping input directories are deduplicated; duplicate logical runs are rejected. With multiple variants of one method, specify `comparison_configs` in the evaluation config so pairs are not guessed.

In [ ]:
import sys
from pathlib import Path

# Usually automatic; set this only if the kernel starts outside the checkout/workspace.
REPO_PATH: str | None = None  # e.g. '/home/mtyrolski/phd/2026/CaRL'
anchors = (Path.cwd(), *Path.cwd().parents, *Path(sys.executable).parents)
candidates = (
    [Path(REPO_PATH).expanduser()]
    if REPO_PATH is not None
    else [candidate for anchor in anchors for candidate in (anchor, anchor / 'CaRL')]
)
ROOT = next((p.resolve() for p in candidates if (p / 'carl/icl').is_dir()), None)
if ROOT is None:
    raise RuntimeError(
        f'Cannot find the CaRL checkout from {Path.cwd()}. '
        'Set REPO_PATH above to the directory containing carl/icl, '
        'and select its .venv/bin/python as the notebook kernel.'
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'CaRL checkout: {ROOT} | Python: {sys.executable}')

# The checkout must be on sys.path before importing its modules.
# pylint: disable=wrong-import-position
import pandas as pd  # noqa: E402 -- checkout path is configured above.
import plotly.graph_objects as go  # noqa: E402 -- checkout path is configured above.
from IPython.display import display  # noqa: E402 -- checkout path is configured above.

from carl.icl.boards import from_rows  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.board_graphics import board_comparison  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.budget_sweep import actual_budget_figure, actual_budget_results  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.data import load_evaluation_config, load_runs  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.diagnostics import board_viewer, calibration_figure, load_calibrations  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.figures import cohort_figures  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.report import build_tables, evaluate  # noqa: E402 -- checkout path is configured above.
from carl.icl.evaluation.statistics import groups  # noqa: E402 -- checkout path is configured above.

# pylint: enable=wrong-import-position

# Default: precomputed offline interface demo, not Luna performance.
CONFIG_PATH = 'configs/icl/evaluation/mock.json'
RUN_DIRS = None  # e.g. ['analysis/luna_icl/runs/YOUR_MATRIX']; None uses config
OUTPUT_DIR = None  # override output directory only if exporting
COHORT_INDEX = None  # None selects the first full-state cohort
config = load_evaluation_config(CONFIG_PATH)
if RUN_DIRS is not None or OUTPUT_DIR is not None:
    config = type(config).model_validate({**config.model_dump(),
        'run_dirs': RUN_DIRS if RUN_DIRS is not None else config.run_dirs,
        'output_dir': OUTPUT_DIR if OUTPUT_DIR is not None else config.output_dir})
print('Input roots:', config.run_dirs)
print('Bootstrap replicates:', config.bootstrap_replicates, '| seed:', config.bootstrap_seed)

## 2. Load outcomes and inspect coverage

The table reports observed and expected runs, missing runs, solved counts and board-balanced completion. Each board has equal weight after averaging its available seeds. API failures remain failures; missing trials are reported separately. A 95% bootstrap resamples boards, keeping their seeds together; no interval is reported for a single board.

In [ ]:
runs = load_runs(config)
tables = build_tables(runs, config)
cohorts = groups(runs)
summary = pd.DataFrame(tables['summary'])
columns = ['cohort', 'config', 'method', 'provider', 'prompt_profile_recorded', 'family', 'easy', 'runs', 'expected_runs',
           'missing_runs', 'boards', 'solved', 'completion', 'ci95_low', 'ci95_high', 'api_errors']
display(summary[columns])
cohort_rows = [{'index': index, 'cohort': key, 'description': values[0].label,
                'conditions': ', '.join(sorted({r.record.config for r in values})), 'runs': len(values)}
               for index, (key, values) in enumerate(cohorts.items())]
display(pd.DataFrame(cohort_rows))

## Solve rate at independently executed budgets

When the input directory contains a budget sweep, this plot uses the outcomes of **separate solver runs at each configured budget**. It is different from the observed-effort curve below, which is calculated from runs at a single cap. Lines connect measured points; they do not guarantee monotonicity. Open markers indicate missing trials, conditions or unstarted planned budgets. Hover to inspect sample counts and API/expansion-budget stops. A completely unstarted budget has no point: compare against `sweep_plan.json`.

`SWEEP_INDEX` selects a matched family/provider/policy contract. Check its table before comparing methods. One board has no 95% interval; repeated seeds are clustered by board. Each budget's paired comparison is available in the exported `pairs.csv`.

In [ ]:
sweep_rows = actual_budget_results(runs, config)
if not sweep_rows:
    print('No comparable multi-budget results in these inputs. Run the budget sweep first.')
else:
    sweep_ids = list(dict.fromkeys(row['sweep'] for row in sweep_rows))
    SWEEP_INDEX = 0
    if not 0 <= SWEEP_INDEX < len(sweep_ids):
        raise ValueError(f'SWEEP_INDEX must be between 0 and {len(sweep_ids) - 1}')
    selected_sweep = [row for row in sweep_rows if row['sweep'] == sweep_ids[SWEEP_INDEX]]
    print('Sweep groups:', sweep_ids, '| selected:', sweep_ids[SWEEP_INDEX])
    display(pd.DataFrame(selected_sweep)[['config', 'method', 'family', 'provider', 'budget',
            'runs', 'expected_runs', 'missing_runs', 'missing_conditions', 'unstarted_budgets', 'boards', 'solved',
            'completion', 'ci95_low', 'ci95_high', 'api_errors', 'api_budget_stops', 'expansion_budget_stops']])
    actual_budget_figure(selected_sweep).show()

## 3. Choose a matched cohort

By default the first full-state cohort is selected; set `COHORT_INDEX` in setup to choose another, then rerun from there. A cohort is the unit for relative/fixed plots; do not pool mock with live, easy fixtures with full boards, or different budgets. Empty paired comparisons are shown explicitly rather than inferred.

In [ ]:
chosen_index = COHORT_INDEX
if chosen_index is None:
    chosen_index = next((i for i, values in enumerate(cohorts.values()) if not values[0].settings.easy), 0)
if not 0 <= chosen_index < len(cohorts):
    raise ValueError(f'COHORT_INDEX must be between 0 and {len(cohorts) - 1}')
cohort_id = list(cohorts)[chosen_index]
print('Selected cohort index:', chosen_index)
selected_runs = cohorts[cohort_id]
selected_tables = {name: [row for row in rows if row['cohort'] == cohort_id] for name, rows in tables.items()}
figures = cohort_figures(selected_runs, selected_tables)
print(cohort_id, '|', selected_runs[0].label)
display(summary[summary['cohort'] == cohort_id][columns])

### Recorded prompt configuration

Prompts and few-shot counts are part of new saved condition configs. Older runs may not contain the prompt profile; such runs are separated from new ones and their missing prompts are not reconstructed. Set `SHOW_SAVED_PROMPTS = True` to inspect instructions that were actually saved.

In [ ]:
SHOW_SAVED_PROMPTS = False
for condition in sorted({run.record.config for run in selected_runs}):
    run = next(run for run in selected_runs if run.record.config == condition)
    recorded = run.detail.get('_prompt_profile_recorded', False)
    print(f'{condition}: model={run.settings.model}, few_shot={run.settings.few_shot}, '
          f'response_format={run.settings.response_format}, prompts_recorded={recorded}')
    if SHOW_SAVED_PROMPTS and recorded:
        display(run.settings.prompts.model_dump())
    elif SHOW_SAVED_PROMPTS:
        print('Prompt profile was not recorded in this legacy run.')

## 4. Completion and observed search budgets

Completion includes all recorded outcomes. The budget curve counts a solved run at its **total attempted search actions**, including failed attempts, not its final plan length. It summarizes runs made with the configured cap; it is not a counterfactual rerun under smaller caps.

In [ ]:
figures['completion'].show()
figures['budget'].show()

## 5. Work, latency and model usage

Attempted actions and wall time include successful and failed runs; solution length is shown only for solved cases. A fast failure is not a better solution. Mock, replay and cached timings do not estimate uncached live latency. Logical usage counts cached inference; provider-attempt counts exclude local cache hits (for mock they are simulated attempts); token usage comes from provider responses. No default dollar prices are assumed.

In [ ]:
figures['effort'].show()
figures['tokens'].show()
figures['calls'].show()
figures['termination'].show()

## 6. Target reachability and paired comparison

The reachability plot separates exact target hits, task completion before the target, and failures for saved completed attempts. An interrupted request may not have an event. Pairs use only matching `(board, seed)`; unmatched trials are counted separately. Difference intervals use board-cluster bootstrap and may be unavailable for small samples.

In [ ]:
figures['reachability'].show()
figures['paired'].show()
display(pd.DataFrame(selected_tables['pairs']))

## 7. Inspect a saved search attempt

Select a run by index. The interactive slider shows the source, requested target and **actual endpoint** for each saved attempt; these can differ. The vector view below shows the first saved target beside its endpoint in the same style as the thesis figures. Runs with no recorded attempts display their initial board instead.

In [ ]:
RUN_INDEX = 0
if not 0 <= RUN_INDEX < len(selected_runs):
    raise ValueError(f'RUN_INDEX must be between 0 and {len(selected_runs) - 1}')
selected = selected_runs[RUN_INDEX]
print(selected.record.config, '|', selected.record.instance, '| seed', selected.record.seed,
      '|', selected.record.finish_reason)
board_viewer(selected, limit=config.max_viewer_attempts).show(auto_play=False)
events = selected.detail.get('events', [])
if events:
    event = events[0]
    print(f"First attempt: scale={event['scale']}, actions={event['attempted_actions']}, exact hit={event['reached']}, solved={event['solved']}")
    board_comparison(from_rows(event['proposal']), from_rows(event['endpoint']),
                     ('Requested target', 'Actual endpoint')).show()
else:
    print('No completed subgoal attempts saved for this run.')

## 8. Calibration diagnostics

The plot uses only calibration files referenced by saved runs. `d` is successful first-hit time, `u` the success-weighted within-context percentile, and `k` the source trajectory offset. Calibration costs are separate from search costs. Missing files are reported, not reconstructed or replaced; fixed-only runs can have no calibration.

In [ ]:
calibrations, missing = load_calibrations(selected_runs)
print(f'Available calibrations: {len(calibrations)} | missing referenced files: {len(missing)}')
for path, calibration in calibrations.items():
    print(path, '| provider:', calibration['provider'], '| retained targets:', len(calibration['examples']))
    calibration_figure(calibration).show()
if missing:
    display(pd.DataFrame({'missing_calibration': missing}))

## 9. Prototype a figure and optionally export

The copied figure can be edited without changing the shared plotting code. Exports are **opt-in**; HTML/JSON remain interactive/editable and static PNG/SVG/PDF require Kaleido. `evaluate(config)` writes the complete offline dashboard, tidy CSV/JSON tables and an input manifest to the configured output directory.

In [ ]:
prototype = go.Figure(figures['completion'])
prototype.update_layout(title='Completion — editable prototype', font={'size': 16}, height=500)
prototype.show()
EXPORT = False  # set True to write the full report and prototype files
if EXPORT:
    dashboard = evaluate(config)
    output = dashboard.parent
    prototype.write_html(output / 'prototype_completion.html', include_plotlyjs='directory')
    prototype.write_json(output / 'prototype_completion.plotly.json')
    # Optional: prototype.write_image(output / 'prototype_completion.svg')
    print('Dashboard:', dashboard)
else:
    print('No files exported. Set EXPORT = True to write the dashboard.')

## Interpretation checklist

- Mock/replay and live outcomes answer different questions; inspect provider and physical usage.
- Missing scheduled runs, API errors, empty attempts and single-board confidence intervals must remain visible.
- Fixed-horizon AdaSubS-style and policy-relative conditions use the same prompted component roles, but different scale labels; fixed cap and shared-cap ablations must be identified separately.
- This explicit-state ICL experiment does not establish the full latent-encoder or learned-generator proposal. See `analysis/luna_icl/PROPOSAL_ALIGNMENT.md` and `configs/icl/evaluation/README.md`.

## Optional: estimate tokens before a live sweep

Set `CALCULATE_TOKEN_PLAN = True` below for an **offline scenario calculation** using prepared boards, locally counted prompt sizes and editable assumptions. Results include calibration once, do not assume cache savings, and are not measurements or billing limits. Price and workload assumptions are in `configs/icl/token_estimates/assumptions.json`; update the price before use. For CSV/JSON/Markdown export use `scripts/estimate_icl_tokens.py` as documented in `configs/icl/budget_sweeps/README.md`.

In [ ]:
CALCULATE_TOKEN_PLAN = False
TOKEN_SWEEP_PATH = 'configs/icl/budget_sweeps/quick.json'  # or pilot.json / transfer.json
if CALCULATE_TOKEN_PLAN:
    from carl.icl.budget_sweep import load_sweep
    from carl.icl.token_estimate import Assumptions, estimate

    token_spec, token_configs = load_sweep(TOKEN_SWEEP_PATH)
    assumptions = Assumptions.model_validate_json(
        (ROOT / 'configs/icl/token_estimates/assumptions.json').read_text(encoding='utf-8')
    )
    token_plan = estimate(token_spec, token_configs, assumptions)
    print('Price snapshot:', token_plan['pricing'])
    print('Configured logical ceilings:', token_plan['logical_limits'])
    display(pd.DataFrame(token_plan['scenarios']))
else:
    print('Offline token calculation skipped; set CALCULATE_TOKEN_PLAN = True to run it.')